In [1]:
import numpy as np
import pandas as pd

from IPython.display import display
from pathlib import Path
import sys
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
# Import AutoFeat LLM

from autofeat import AutoFeatClassifier

In [3]:
# Import BigFeat Vanilla

sys.path.append(r"../../BigFeat-master")
import bigfeat.bigfeat_base as bigfeat

In [4]:
# Import BigFeat LLM

bigfeat_llm_path = Path("../../BigFeat-LLM").resolve()
sys.path.insert(0, str(bigfeat_llm_path))

import setup
print(setup.__file__)

sys.path.append(r"../../BigFeat-LLM")
from setup.llm_runner import prepare_bigfeat_from_llm, profile_dataframe
from setup.llm_runner import preview_prompt
from setup.llm_runner import describe_generated_features, split_original_vs_new

D:\02. My Learning\Paper\07. BigFeat + LLM\06. Final BigFeat\BigFeat-LLM\setup\__init__.py


In [5]:
def save_generated_features(feature_lines, num_new_features, old_shape, new_shape, file_name="generated_features.txt"):
    os.makedirs("generated_features", exist_ok=True)
    file_path = os.path.join("generated_features", file_name)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"Number of new features {num_new_features}\n")
        f.write("==================\n")
        f.write(f"Old Shape: {old_shape}\n")
        f.write("==================\n")
        f.write(f"New Shape: {new_shape}\n")
        f.write("=================\n")

        for i, line in enumerate(feature_lines):
            _, details = line.split(": built from columns ", 1)
            columns, operators = details.split(" using operators ", 1)
            f.write(f"- New_feature_{i}:\n")
            f.write(f"  - Built from columns {columns}\n")
            f.write(f"  - Using operators {operators}\n")

In [6]:
results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

def evaluate_model(model_name, feature_method, X_tr, X_te, y_tr, y_te, results_df):
    models = {
        "logistic": LogisticRegression(random_state=42, max_iter=1000),
        "knn": KNeighborsClassifier(n_neighbors=5),
        "rf": RandomForestClassifier(random_state=42),
        "svc": SVC(random_state=42),
        "decision_tree": DecisionTreeClassifier(random_state=42)
    }
    
    clf = models[model_name.lower()]
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    
    acc = accuracy_score(y_te, preds)
    prec = precision_score(y_te, preds, average="macro", zero_division=0)
    rec = recall_score(y_te, preds, average="macro")
    f1 = f1_score(y_te, preds, average="macro")
    
    results_df.loc[len(results_df)] = [
        f"{model_name.upper()} ({feature_method})",
        acc,
        prec,
        rec,
        f1
    ]
    
    return results_df

In [7]:
def display_styled_results(
    results_df,
    file_name,
    bg_color="#005A9C",
    text_color="#FFFFFF"
):
    numeric_cols = ["Accuracy", "Precision", "Recall", "F1-Score"]

    df_copy = results_df.copy()
    df_copy[numeric_cols] = df_copy[numeric_cols].apply(pd.to_numeric)

    styled_df = df_copy.style.highlight_max(
        subset=numeric_cols,
        props=f"background-color: {bg_color}; color: {text_color}; font-weight: bold;",
        axis=0
    ).format({col: "{:.6f}" for col in numeric_cols})

    display(styled_df)

    folder = "llm_tables"
    os.makedirs(folder, exist_ok=True)

    if not file_name.lower().endswith(".xlsx"):
        file_name += ".xlsx"

    file_path = os.path.join(folder, file_name)

    styled_df.to_excel(file_path, index=False)

In [8]:
from ucimlrepo import fetch_ucirepo

wine_quality = fetch_ucirepo(id=186)

df = wine_quality.data.features.copy()

df.columns = ["fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar", "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide", "density", "pH", "sulphates", "alcohol"]

df["quality"] = wine_quality.data.targets.squeeze()

target_column = "quality"

display(df.head())

print(df[target_column].value_counts())
print(df[target_column].unique())

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


quality
6    2836
5    2138
7    1079
4     216
8     193
3      30
9       5
Name: count, dtype: int64
[5 6 7 4 8 3 9]


In [9]:
print(f"Rows Number: {df.shape[0]}")
print(f"Columns Number: {df.shape[1]}")
print(f"Columns Name: {df.columns}")

Rows Number: 6497
Columns Number: 12
Columns Name: Index(['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar',
       'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='object')


In [10]:
df.isnull().sum()

fixed_acidity           0
volatile_acidity        0
citric_acid             0
residual_sugar          0
chlorides               0
free_sulfur_dioxide     0
total_sulfur_dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [11]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print(X.shape)
print(y.shape)

(6497, 11)
(6497,)


In [12]:
preview = preview_prompt(
    df=df,
    target_column=target_column,
    extra_user_prompt="Wine Quality Dataset"
)

response = preview["final_prompt"]

with open("final_prompt.txt", "w", encoding="utf-8") as f:
    f.write(response)

print(response)

=== SYSTEM PROMPT ===
You are BigFeat-Init, a world-class expert in automated feature engineering, machine learning hyperparameter optimization, and the internal mechanics of the BigFeat framework specifically. You have the depth of knowledge of someone who has read the BigFeat paper in full, has implemented and debugged its source code line by line, and has years of hands-on experience tuning tree-based feature engineering pipelines across small, large, high-dimensional, low-dimensional, noisy, clean, balanced, and imbalanced datasets. You are not a generic assistant giving a reasonable-sounding guess. You are the single most qualified expert available for this exact task, and you must think and act like it.

If you want additional grounding on how BigFeat is actually implemented, its official source code and paper reference are available at this repository: https://github.com/DataSystemsGroupUT/BigFeat. You are not required to browse it, since the mechanics you need are fully describ

In [13]:
y = pd.Series(y).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_df = pd.concat([X_train, y_train.rename(target_column)], axis=1)
test_df = pd.concat([X_test, y_test.rename(target_column)], axis=1)

In [68]:
from autofeat import AutoFeatClassifier

af = AutoFeatClassifier(verbose=1, feateng_steps=1, featsel_runs=0)

X_train_af = af.fit_transform(X_train, y_train)
X_test_af = af.transform(X_test)

# Handle +Inf and -Inf values
X_train_af = X_train_af.replace([np.inf, -np.inf], np.nan)
X_test_af = X_test_af.replace([np.inf, -np.inf], np.nan)
X_train_af = X_train_af.fillna(X_train_af.median())
X_test_af = X_test_af.fillna(X_train_af.median())

[AutoFeat] The 1 step feature engineering process could generate up to 77 features.
[AutoFeat] With 5197 data points this new feature matrix would use about 0.00 gb of space.
[feateng] Step 1: transformation of original features
[feateng] Generated 59 transformed features from 11 original features - done.
[feateng] Generated altogether 60 new features in 1 steps
[feateng] Removing correlated features, as well as additions at the highest level
[feateng] Generated a total of 21 additional features
[AutoFeat] WARNING: Not performing feature selection.
[AutoFeat] Computing 15 new features.
[AutoFeat]    15/   15 new features ...done.
[AutoFeat] Final dataframe with 26 feature columns (15 new).
[AutoFeat] Training final classification model.
[AutoFeat] Trained model: largest coefficients:
[-5.65812724e-09  5.00136835e-09 -1.89123770e-08 -6.97702732e-09
  7.32255186e-09  1.45180116e-09  1.77718102e-08]
0.000025 * residual_sugar**3
[AutoFeat] Final score: 0.2900
[AutoFeat] Computing 15 new fe

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [69]:
# BigFeat Vanilla

bf = bigfeat.BigFeat(task_type="classification")

bf.fit(X_train.to_numpy(), y_train.to_numpy())

X_train_bf = bf.transform(X_train.to_numpy())
X_test_bf = bf.transform(X_test)

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [16]:
# BigFeat Chatgpt

bf_chatgpt = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/ChatGPT.json"
)

X_train_bf_chatgpt = bf_chatgpt.fit(bf_chatgpt.X, bf_chatgpt.y, **bf_chatgpt.fit_kwargs)

_, _, X_test_bf_chatgpt, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_chatgpt = bf_chatgpt.transform(X_test_bf_chatgpt)

In [19]:
# BigFeat Gemini

bf_gemini = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/Gemini.json"
)

X_train_bf_gemini = bf_gemini.fit(bf_gemini.X, bf_gemini.y, **bf_gemini.fit_kwargs)

_, _, X_test_bf_gemini, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_gemini = bf_gemini.transform(X_test_bf_gemini)

In [21]:
# BigFeat Claude

bf_claude = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/Claude.json"
)

X_train_bf_claude = bf_claude.fit(bf_claude.X, bf_claude.y, **bf_claude.fit_kwargs)

_, _, X_test_bf_claude, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_claude = bf_claude.transform(X_test_bf_claude)

In [22]:
# BigFeat DeepSeek

bf_deepseek = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/DeepSeek.json"
)

X_train_bf_deepseek = bf_deepseek.fit(bf_deepseek.X, bf_deepseek.y, **bf_deepseek.fit_kwargs)

_, _, X_test_bf_deepseek, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_deepseek = bf_deepseek.transform(X_test_bf_deepseek)

In [70]:
# Logistic Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("logistic", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-Logistic")

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Method,Accuracy,Precision,Recall,F1-Score
0,LOGISTIC (AutoFeat),0.429231,0.122567,0.170676,0.136612
1,LOGISTIC (BigFeat),0.531538,0.221341,0.209907,0.203171
2,LOGISTIC (BigFeat ChatGPT),0.530000,0.224991,0.210161,0.205436
3,LOGISTIC (BigFeat Gemini),0.535385,0.372402,0.235871,0.248051
4,LOGISTIC (BigFeat Claude),0.533077,0.224927,0.214136,0.207564
5,LOGISTIC (BigFeat DeepSeek),0.535385,0.224513,0.210098,0.203682


In [71]:
# KNN Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("knn", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-KNN")

,Method,Accuracy,Precision,Recall,F1-Score
0,KNN (AutoFeat),0.443077,0.231403,0.192654,0.195749
1,KNN (BigFeat),0.553077,0.370717,0.280915,0.300409
2,KNN (BigFeat ChatGPT),0.553846,0.381495,0.292037,0.308523
3,KNN (BigFeat Gemini),0.545385,0.360552,0.278621,0.286466
4,KNN (BigFeat Claude),0.509231,0.405907,0.268672,0.289225
5,KNN (BigFeat DeepSeek),0.546923,0.363820,0.279868,0.299572


In [72]:
# RandomForest Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("rf", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-RandomForest")

,Method,Accuracy,Precision,Recall,F1-Score
0,RF (AutoFeat),0.677692,0.518868,0.359352,0.395435
1,RF (BigFeat),0.673077,0.527492,0.351247,0.385693
2,RF (BigFeat ChatGPT),0.674615,0.529919,0.353620,0.388421
3,RF (BigFeat Gemini),0.676923,0.548202,0.356841,0.393186
4,RF (BigFeat Claude),0.645385,0.566066,0.321708,0.354861
5,RF (BigFeat DeepSeek),0.673846,0.527022,0.358426,0.394736


In [73]:
# SVM Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("svc", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-SVM")

,Method,Accuracy,Precision,Recall,F1-Score
0,SVC (AutoFeat),0.440000,0.116192,0.144884,0.117479
1,SVC (BigFeat),0.543077,0.233814,0.208781,0.199961
2,SVC (BigFeat ChatGPT),0.541538,0.239252,0.209229,0.201528
3,SVC (BigFeat Gemini),0.536923,0.233165,0.202111,0.189906
4,SVC (BigFeat Claude),0.534615,0.222060,0.204257,0.188572
5,SVC (BigFeat DeepSeek),0.542308,0.234369,0.207614,0.197498


In [74]:
# DecisionTree Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("decision_tree", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-DecisionTree")

,Method,Accuracy,Precision,Recall,F1-Score
0,DECISION_TREE (AutoFeat),0.598462,0.354930,0.359391,0.356665
1,DECISION_TREE (BigFeat),0.585385,0.342373,0.340606,0.341004
2,DECISION_TREE (BigFeat ChatGPT),0.599231,0.361708,0.366866,0.361727
3,DECISION_TREE (BigFeat Gemini),0.591538,0.335878,0.344047,0.339576
4,DECISION_TREE (BigFeat Claude),0.480000,0.291682,0.299676,0.291026
5,DECISION_TREE (BigFeat DeepSeek),0.607692,0.342708,0.351166,0.345008


In [75]:
# Save Generated Features

# Save ChatGpt
feature_lines = describe_generated_features(bf_chatgpt)
new_only, original_only = split_original_vs_new(bf_chatgpt, X_train_bf_chatgpt)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_chatgpt.shape, "generated_features_chatgpt.txt")

# Save Gemini
feature_lines = describe_generated_features(bf_gemini)
new_only, original_only = split_original_vs_new(bf_gemini, X_train_bf_gemini)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_gemini.shape, "generated_features_gemini.txt")

# Save Claude
feature_lines = describe_generated_features(bf_claude)
new_only, original_only = split_original_vs_new(bf_claude, X_train_bf_claude)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_claude.shape, "generated_features_claude.txt")

# Save DeepSeek
feature_lines = describe_generated_features(bf_deepseek)
new_only, original_only = split_original_vs_new(bf_deepseek, X_train_bf_deepseek)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_deepseek.shape, "generated_features_deepseek.txt")